# Analysis 1: 오르카 글로벌 전환 퍼널 분석 계획 및 1차 결과

이 노트북은 우리가 정의한 `funnel_stage`, `lifecycle_status`, `identity_level` 기준으로 글로벌 유입-예약-결제 흐름을 분석하기 위한 1차 분석 파일

## 목표

- 국가/언어/채널/캠페인별 유입 구조 확인
- 유입부터 예약 생성, 결제 완료까지의 전환 퍼널 확인
- 특정 단계에서 예약률/결제 완료율 차이가 나는 구간 식별
- 보험 선택, 채널, 캠페인, 언어별 전환 차이 확인
- 다음 스프린트에서 볼 지표와 개선 액션 우선순위 도출

## 현재 핵심 요약

- canonical user: 68,814개
- reservation: 26,921건
- reservation user: 10,861개 (15.8%)
- paid user: 4,139개 (6.0%)
- paid reservation: 4,555건 (16.9%)


# Analysis 1 기준 업데이트

이번 버전은 `identity_level = reservation_id`인 1,176개 canonical unit을 유저 퍼널 분석에서 제외합니다. 이유는 customer_id 없이 reservation_id만 있는 미식별 예약 단위라, 사용자 단위 전환율을 왜곡할 수 있기 때문입니다.

- 원본 canonical user: 68,814
- 제외: 1,176
- 분석 대상 canonical user: 67,638
- 예약 생성 유저: 9,685
- 결제 완료 유저: 2,963



## 0. 분석 기준

이번 분석에서는 좋은 예약/좋은 고객을 복잡한 품질 점수로 보지 않고, 예약 생성과 결제 완료 중심으로 정의합니다.

- 좋은 예약: `ops_reservation_id`가 생성된 예약
- 좋은 고객: 예약을 생성한 고객 또는 canonical user
- 전환 완료: `payment_status = PAID`
- 취소/환불/반납 여부: 전환 완료 여부가 아니라 후속 lifecycle 상태로 분리

퍼널은 아래 순서의 누적 도달 단계로 봅니다.

```text
anonymous_visit
→ identified_visit
→ search_started
→ car_clicked
→ booking_started
→ insurance_viewed
→ insurance_selected
→ checkout_started
→ payment_attempted
→ payment_completed
```


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('/Users/yangjiyun/Desktop/LYK')
PROCESSED = BASE / 'data' / 'processed'
ANALYSIS = BASE / 'analysis1'

user = pd.read_csv(PROCESSED / 'user_funnel_state.csv', low_memory=False)
res = pd.read_csv(PROCESSED / 'reservation_funnel_state.csv', low_memory=False)

stage_order = [
    'anonymous_visit', 'identified_visit', 'search_started', 'car_clicked',
    'booking_started', 'insurance_viewed', 'insurance_selected',
    'checkout_started', 'payment_attempted', 'payment_completed'
]
stage_rank = {s: i + 1 for i, s in enumerate(stage_order)}

user['funnel_stage_rank'] = pd.to_numeric(user['funnel_stage_rank'], errors='coerce').fillna(0).astype(int)
user['reservation_count'] = pd.to_numeric(user['reservation_count'], errors='coerce').fillna(0).astype(int)
user['paid_reservation_count'] = pd.to_numeric(user['paid_reservation_count'], errors='coerce').fillna(0).astype(int)
user['has_reservation'] = user['reservation_count'] > 0
user['has_paid'] = user['paid_reservation_count'] > 0

res['final_price_base'] = pd.to_numeric(res['final_price_base'], errors='coerce')
res['has_paid'] = res['payment_status'].eq('PAID')

print(user.shape, res.shape)


## 1. 전체 누적 퍼널

`user_funnel_state.csv`의 `funnel_stage`는 각 canonical user가 도달한 가장 깊은 단계입니다. 단계별 퍼널은 `funnel_stage_rank >= 해당 단계 rank` 방식으로 누적 계산했습니다.

| stage | users_reached | overall_reach_rate | step_conversion_rate | dropoff_from_prev |
| --- | --- | --- | --- | --- |
| anonymous_visit | 68,814 | 100.0% | 100.0% | 0.0% |
| identified_visit | 20,557 | 29.9% | 29.9% | 70.1% |
| search_started | 20,394 | 29.6% | 99.2% | 0.8% |
| car_clicked | 16,572 | 24.1% | 81.3% | 18.7% |
| booking_started | 10,861 | 15.8% | 65.5% | 34.5% |
| insurance_viewed | 10,861 | 15.8% | 100.0% | 0.0% |
| insurance_selected | 10,861 | 15.8% | 100.0% | 0.0% |
| checkout_started | 10,861 | 15.8% | 100.0% | 0.0% |
| payment_attempted | 4,315 | 6.3% | 39.7% | 60.3% |
| payment_completed | 4,139 | 6.0% | 95.9% | 4.1% |

### 1차 해석

- 전체 canonical user 중 상당수는 익명 방문 단계에 머뭅니다.
- `checkout_started`와 `payment_completed`는 예약/결제 DB와 연결되는 핵심 하단 퍼널입니다.
- 단계별 전환율은 identity 수준을 분리해서 다시 봐야 합니다. 익명 세션이 많기 때문에 전체 전환율만 보면 과소평가될 수 있습니다.


In [ ]:
rows = []
for stage, rank in stage_rank.items():
    rows.append({
        'stage': stage,
        'rank': rank,
        'users_reached': int((user['funnel_stage_rank'] >= rank).sum())
    })
funnel = pd.DataFrame(rows)
funnel['prev_users'] = funnel['users_reached'].shift(1)
funnel['step_conversion_rate'] = funnel['users_reached'] / funnel['prev_users']
funnel.loc[0, 'step_conversion_rate'] = 1.0
funnel['overall_reach_rate'] = funnel['users_reached'] / len(user)
funnel['dropoff_from_prev'] = 1 - funnel['step_conversion_rate']
display(funnel)

plt.figure(figsize=(10, 4))
plt.bar(funnel['stage'], funnel['users_reached'])
plt.xticks(rotation=45, ha='right')
plt.title('Cumulative funnel users by stage')
plt.ylabel('canonical users')
plt.tight_layout()
plt.show()


## 2. identity_level별 유저 분포

`identity_level`은 같은 유저로 묶은 기준의 신뢰도입니다.

| identity_level | canonical_users |
| --- | --- |
| anonymous_session | 57,615 |
| customer_id | 10,023 |
| reservation_id | 1,176 |

### 해석

- `customer_id` 기준은 신뢰도가 가장 높지만 표본이 줄어듭니다.
- `anonymous_session`은 로그인 전 행동을 넓게 볼 수 있지만 사용자 단위라기보다 세션 단위에 가깝습니다.
- 전환율 리포트는 전체와 `customer_id` 기준을 병렬로 보는 것이 안전합니다.


In [ ]:
identity_summary = user.groupby('identity_level').agg(
    canonical_users=('canonical_user_key', 'count'),
    reservation_users=('has_reservation', 'sum'),
    paid_users=('has_paid', 'sum')
).reset_index()
identity_summary['reservation_rate'] = identity_summary['reservation_users'] / identity_summary['canonical_users']
identity_summary['paid_user_rate'] = identity_summary['paid_users'] / identity_summary['canonical_users']
identity_summary['paid_after_reservation_rate'] = identity_summary['paid_users'] / identity_summary['reservation_users'].replace(0, pd.NA)
display(identity_summary.sort_values('canonical_users', ascending=False))


## 3. 채널별 예약/결제 성과

`utm_source_first` 기준 상위 채널입니다. unknown은 유입 정보가 비어 있거나 세션/예약 연결 과정에서 확보되지 않은 케이스입니다.

| utm_source_first | canonical_users | reservation_users | paid_users | reservation_rate | paid_user_rate | paid_after_reservation_rate |
| --- | --- | --- | --- | --- | --- | --- |
| (unknown) | 34,680 | 8,951 | 3,386 | 25.8% | 9.8% | 37.8% |
| google | 29,007 | 1,284 | 387 | 4.4% | 1.3% | 30.1% |
| app | 1,753 | 570 | 330 | 32.5% | 18.8% | 57.9% |
| INFLUENCER | 34 | 13 | 10 | 38.2% | 29.4% | 76.9% |
| {{site_source_name}} | 185 | 10 | 6 | 5.4% | 3.2% | 60.0% |
| whatsapp | 13 | 6 | 6 | 46.2% | 46.2% | 100.0% |
| BFF | 69 | 6 | 5 | 8.7% | 7.2% | 83.3% |
| shuttle-blog | 31 | 10 | 5 | 32.3% | 16.1% | 50.0% |
| ig | 316 | 6 | 2 | 1.9% | 0.6% | 33.3% |
| fb | 2,547 | 4 | 1 | 0.2% | 0.0% | 25.0% |
| email | 22 | 1 | 1 | 4.5% | 4.5% | 100.0% |
| chatgpt.com | 52 | 0 | 0 | 0.0% | 0.0% |  |
| superalink | 40 | 0 | 0 | 0.0% | 0.0% |  |
| trustpilot | 33 | 0 | 0 | 0.0% | 0.0% |  |
| tt-ads | 8 | 0 | 0 | 0.0% | 0.0% |  |

### 볼 포인트

- 방문/세션 수가 많은 채널과 실제 예약/결제 완료를 만드는 채널이 다를 수 있습니다.
- `reservation_rate`는 예약 생성까지의 효율, `paid_user_rate`는 결제 완료까지의 효율입니다.
- 예산 배분 판단에는 단순 유입 수보다 결제 완료율과 예약 수를 함께 봐야 합니다.


In [ ]:
def summarize_user_by(col):
    x = user.copy()
    x[col] = x[col].fillna('').replace('', '(unknown)')
    out = x.groupby(col).agg(
        canonical_users=('canonical_user_key','count'),
        reservation_users=('has_reservation','sum'),
        paid_users=('has_paid','sum'),
        avg_event_count=('event_count','mean'),
    ).reset_index()
    out['reservation_rate'] = out['reservation_users'] / out['canonical_users']
    out['paid_user_rate'] = out['paid_users'] / out['canonical_users']
    out['paid_after_reservation_rate'] = out['paid_users'] / out['reservation_users'].replace(0, pd.NA)
    return out.sort_values(['paid_users','canonical_users'], ascending=False)

channel_summary = summarize_user_by('utm_source_first')
display(channel_summary.head(20))


## 4. Medium / Campaign / Language별 차이

### Medium 상위

| utm_medium_first | canonical_users | reservation_users | paid_users | reservation_rate | paid_user_rate |
| --- | --- | --- | --- | --- | --- |
| (unknown) | 36,483 | 9,485 | 3,690 | 26.0% | 10.1% |
| cpc | 16,655 | 1,306 | 402 | 7.8% | 2.4% |
| INFLUENCER | 41 | 20 | 16 | 48.8% | 39.0% |
| {{placement}} | 187 | 12 | 8 | 6.4% | 4.3% |
| referral | 71 | 8 | 7 | 11.3% | 9.9% |
| blog | 25 | 10 | 5 | 40.0% | 20.0% |
| retargeting | 7 | 4 | 4 | 57.1% | 57.1% |
| social | 25 | 3 | 2 | 12.0% | 8.0% |
| crm | 6 | 2 | 2 | 33.3% | 33.3% |
| paid | 2,699 | 6 | 1 | 0.2% | 0.0% |
| link | 38 | 2 | 1 | 5.3% | 2.6% |
| marketing | 22 | 1 | 1 | 4.5% | 4.5% |
| demandgen | 12,379 | 2 | 0 | 0.0% | 0.0% |
| Facebook_Right_Column | 132 | 0 | 0 | 0.0% | 0.0% |
| company_profile | 36 | 0 | 0 | 0.0% | 0.0% |

### Campaign 상위

| utm_campaign_first | canonical_users | reservation_users | paid_users | reservation_rate | paid_user_rate |
| --- | --- | --- | --- | --- | --- |
| (unknown) | 36,132 | 9,485 | 3,691 | 26.3% | 10.2% |
| 23754383997 | 2,998 | 446 | 159 | 14.9% | 5.3% |
| 23735454208 | 2,677 | 326 | 122 | 12.2% | 4.6% |
| 23205597293 | 1,470 | 201 | 59 | 13.7% | 4.0% |
| 23688781100 | 2,506 | 261 | 34 | 10.4% | 1.4% |
| 23683448292 | 612 | 70 | 28 | 11.4% | 4.6% |
| INFLUENCER | 41 | 20 | 16 | 48.8% | 39.0% |
| {{campaign.name}} | 187 | 12 | 8 | 6.4% | 4.3% |
| BFF_program | 71 | 8 | 7 | 11.3% | 9.9% |
| shuttle-partnership | 29 | 10 | 5 | 34.5% | 17.2% |
| abandoned_reservation_v2 | 7 | 4 | 4 | 57.1% | 57.1% |
| referral_share_25 | 6 | 2 | 2 | 33.3% | 33.3% |
| 120244167499720340 | 155 | 5 | 1 | 3.2% | 0.6% |
| influencer | 38 | 2 | 1 | 5.3% | 2.6% |
| family-special-2026-05 | 22 | 1 | 1 | 4.5% | 4.5% |

### Language 상위

| language_first | canonical_users | reservation_users | paid_users | reservation_rate | paid_user_rate |
| --- | --- | --- | --- | --- | --- |
| (unknown) | 8,190 | 8,190 | 3,128 | 100.0% | 38.2% |
| en | 40,192 | 1,922 | 807 | 4.8% | 2.0% |
| zh | 9,327 | 537 | 114 | 5.8% | 1.2% |
| de | 325 | 26 | 14 | 8.0% | 4.3% |
| ko | 2,152 | 25 | 13 | 1.2% | 0.6% |
| fr | 400 | 28 | 12 | 7.0% | 3.0% |
| es | 331 | 28 | 11 | 8.5% | 3.3% |
| nl | 145 | 19 | 10 | 13.1% | 6.9% |
| it | 119 | 18 | 7 | 15.1% | 5.9% |
| ja | 219 | 10 | 5 | 4.6% | 2.3% |
| ru | 306 | 13 | 4 | 4.2% | 1.3% |
| da | 31 | 6 | 4 | 19.4% | 12.9% |
| pt | 79 | 7 | 3 | 8.9% | 3.8% |
| ar | 236 | 2 | 1 | 0.8% | 0.4% |
| ca | 14 | 1 | 1 | 7.1% | 7.1% |

### 볼 포인트

- 국가/언어/캠페인별로 어느 단계에서 이탈하는지 비교해야 합니다.
- 언어 값이 비어 있는 사용자가 많으면 Mixpanel 위치/언어 속성으로 보완하는 방안을 검토합니다.


In [ ]:
medium_summary = summarize_user_by('utm_medium_first')
campaign_summary = summarize_user_by('utm_campaign_first')
language_summary = summarize_user_by('language_first')

display(medium_summary.head(20))
display(campaign_summary.head(20))
display(language_summary.head(20))


## 5. 보험 선택과 전환

보험은 외국인 고객의 불안 요소와 가격 민감도가 함께 걸리는 구간입니다. 이벤트 기준 보험 선택과 DB 보험 row 존재 여부를 분리해서 봤습니다.

| has_insurance_selected_event | has_reservation_insurance_row | canonical_users | reservation_users | paid_users | reservation_rate | paid_user_rate | paid_after_reservation_rate |
| --- | --- | --- | --- | --- | --- | --- | --- |
| False | False | 58,996 | 1,043 | 946 | 1.8% | 1.6% | 90.7% |
| True | False | 16 | 16 | 0 | 100.0% | 0.0% | 0.0% |
| False | True | 8,177 | 8,177 | 2,577 | 100.0% | 31.5% | 31.5% |
| True | True | 1,625 | 1,625 | 616 | 100.0% | 37.9% | 37.9% |

### 볼 포인트

- `has_insurance_selected_event`는 행동 이벤트입니다.
- `has_reservation_insurance_row`는 실제 예약에 연결된 보험 DB row입니다.
- 두 값이 같은 사실은 아니므로, 보험 화면에서의 선택 행동과 실제 예약 보험 행을 구분해야 합니다.


In [ ]:
insurance_summary = user.groupby(['has_insurance_selected_event','has_reservation_insurance_row']).agg(
    canonical_users=('canonical_user_key','count'),
    reservation_users=('has_reservation','sum'),
    paid_users=('has_paid','sum'),
    refunds=('refund_count','sum')
).reset_index()
insurance_summary['reservation_rate'] = insurance_summary['reservation_users'] / insurance_summary['canonical_users']
insurance_summary['paid_user_rate'] = insurance_summary['paid_users'] / insurance_summary['canonical_users']
insurance_summary['paid_after_reservation_rate'] = insurance_summary['paid_users'] / insurance_summary['reservation_users'].replace(0, pd.NA)
display(insurance_summary)


## 6. 예약 단위 lifecycle / 결제 상태

사용자 퍼널과 별도로 예약 단위에서는 예약 상태와 결제 상태를 봅니다.

### 예약 lifecycle 분포

| reservation_lifecycle_status | reservations |
| --- | --- |
| payment_pending | 22,074 |
| returned | 3,482 |
| refunded | 664 |
| cancelled | 253 |
| reserved | 226 |
| partially_refunded | 200 |
| in_use | 20 |
| payment_failed | 1 |
| paid | 1 |

### 결제 상태 분포

| payment_status | reservations |
| --- | --- |
| PENDING | 22,078 |
| PAID | 4,555 |
| REFUNDED | 284 |
| PARTIALLY_REFUNDED | 4 |

### 해석

- `payment_status = PAID`가 최종 전환 완료 기준입니다.
- `lifecycle_status`는 최종 종료 상태가 아니라 데이터 추출 시점 기준의 최신 예약/결제 상태입니다.
- `reserved`, `in_use`는 진행 중 상태로 볼 수 있습니다.


In [ ]:
display(res['lifecycle_status'].value_counts().rename_axis('lifecycle_status').reset_index(name='reservations'))
display(res['payment_status'].value_counts().rename_axis('payment_status').reset_index(name='reservations'))


## 7. 예약 기준 채널 성과

예약 row 기준으로 채널별 예약 수, 결제 완료 수, 매출, 취소/환불 상태를 봅니다.

| utm_source | reservations | paid_reservations | paid_reservation_rate | revenue_base | avg_revenue_base | refund_or_partial_rate | cancelled_rate |
| --- | --- | --- | --- | --- | --- | --- | --- |
| (unknown) | 26,854 | 4,543 | 16.9% | 837,388,954 | 31,183 | 3.2% | 0.9% |
| google | 54 | 9 | 16.7% | 2,839,371 | 52,581 | 7.4% | 0.0% |
| {{site_source_name}} | 4 | 2 | 50.0% | 27,839 | 6,960 | 0.0% | 50.0% |
| app | 8 | 1 | 12.5% | 214,881 | 26,860 | 0.0% | 0.0% |
| INFLUENCER | 1 | 0 | 0.0% | 8,049 | 8,049 | 0.0% | 0.0% |

### 볼 포인트

- 유저 기준 채널 성과와 예약 기준 채널 성과를 같이 봐야 합니다.
- 예약을 많이 만드는 채널과 결제 완료/매출 기여가 높은 채널은 다를 수 있습니다.


In [ ]:
res_channel = res.copy()
res_channel['utm_source'] = res_channel['utm_source'].fillna('').replace('', '(unknown)')
res_channel_summary = res_channel.groupby('utm_source').agg(
    reservations=('ops_reservation_id','count'),
    paid_reservations=('has_paid','sum'),
    revenue_base=('final_price_base','sum'),
    avg_revenue_base=('final_price_base','mean'),
    refunded=('lifecycle_status', lambda s: s.isin(['refunded','partially_refunded']).sum()),
    cancelled=('lifecycle_status', lambda s: s.eq('cancelled').sum()),
).reset_index()
res_channel_summary['paid_reservation_rate'] = res_channel_summary['paid_reservations'] / res_channel_summary['reservations']
res_channel_summary['refund_or_partial_rate'] = res_channel_summary['refunded'] / res_channel_summary['reservations']
res_channel_summary['cancelled_rate'] = res_channel_summary['cancelled'] / res_channel_summary['reservations']
display(res_channel_summary.sort_values(['paid_reservations','reservations'], ascending=False).head(20))


## 8. 이 분석으로 줄 수 있는 인사이트 형태

이 분석은 아래 질문에 답하기 위한 출발점입니다.

1. 어떤 채널이 단순 방문이 아니라 예약 생성까지 이어지는가?
2. 어떤 채널이 결제 완료까지 이어지는가?
3. 어떤 구간에서 가장 큰 이탈이 발생하는가?
4. 보험 조회/선택이 예약 또는 결제 전환에 어떤 차이를 만드는가?
5. 언어/캠페인/채널별로 예약률과 결제 완료율이 어떻게 다른가?
6. 예약은 만들지만 결제 완료율이 낮은 구간 또는 채널은 어디인가?
7. 결제 완료 이후 취소/환불은 어떤 채널에서 더 많이 발생하는가?

## 다음 액션아이템

- 전체 Mixpanel event export를 확보해 `Distinct ID → $distinct_id → customer_id` 연결을 재검증합니다.
- CS 문의 유형을 예약 전환과 연결하는 `cs_conversion_summary`를 만듭니다.
- 국가/언어 정보 보완을 위해 Mixpanel 위치 속성과 내부 이벤트 속성의 bridge 정책을 정합니다.
- 채널/캠페인별 단계 전환율을 대시보드 형태로 만듭니다.
